# Aim 2: Temporal Trends in Psychiatric Comorbidities (2011-2020)

**Project:** Prevalence and Trends of DSM-5 Axis I Psychiatric Disorders in Epilepsy Surgery  
**Data:** Analytic cohort from `01_cohort_building.ipynb`

This notebook:
1. Annual survey-weighted prevalence of psychiatric disorders
2. Trend analysis (survey-weighted logistic with YEAR as continuous)
3. Stratified trends: surgery vs non-surgical
4. Line plots with 95% CI bands
5. Surgery utilization trends over time

In [ ]:
# ============================================================
# SETUP
# ============================================================
library(arrow)
library(data.table)
library(survey)
library(ggplot2)
library(scales)
library(gridExtra)

options(survey.lonely.psu = "adjust")

outdir <- "/Volumes/Niels 2/NIS_new_version/NIS_epilepsy_psych"

cat("Loading analytic cohort...\n")
dt <- as.data.table(read_parquet(file.path(outdir, "output/epilepsy_analytic.parquet")))

# Prepare
dt[, group := fifelse(any_surgery, "Surgery", "Non-Surgical")]
psych_cols <- grep("^psych_", names(dt), value = TRUE)
for (col in c(psych_cols, "any_psych")) {
    dt[[col]] <- as.integer(dt[[col]])
}

# Survey design
svy <- svydesign(id = ~HOSP_NIS, strata = ~NIS_STRATUM, weights = ~DISCWT, nest = TRUE, data = dt)

cat(sprintf("Loaded: %s rows, years: %s\n",
            format(nrow(dt), big.mark = ","),
            paste(sort(unique(dt$YEAR)), collapse = ", ")))
invisible(NULL)

In [ ]:
# ============================================================
# ANNUAL PREVALENCE ESTIMATES (survey-weighted)
# ============================================================
psych_labels <- c(
    "psych_depression" = "Depression", "psych_bipolar" = "Bipolar",
    "psych_anxiety" = "Anxiety", "psych_ptsd" = "PTSD",
    "psych_ocd" = "OCD", "psych_schizophrenia" = "Schizophrenia",
    "psych_psychosis" = "Psychosis", "psych_adhd" = "ADHD",
    "psych_alcohol" = "Alcohol", "psych_drug" = "Drug",
    "psych_suicidal" = "Suicidal ideation", "psych_pnes" = "PNES",
    "psych_organic" = "Organic", "any_psych" = "Any psychiatric"
)

years <- sort(unique(dt$YEAR))

# Compute annual prevalence for each disorder, overall and by group
annual <- data.frame()
for (y in years) {
    svy_y <- subset(svy, YEAR == y)
    svy_y_s <- subset(svy, YEAR == y & group == "Surgery")
    svy_y_ns <- subset(svy, YEAR == y & group == "Non-Surgical")

    for (col in names(psych_labels)) {
        fmla <- as.formula(paste0("~", col))

        for (grp_info in list(
            list(name = "Overall", d = svy_y),
            list(name = "Surgery", d = svy_y_s),
            list(name = "Non-Surgical", d = svy_y_ns)
        )) {
            m <- tryCatch(svymean(fmla, grp_info$d, na.rm = TRUE), error = function(e) NULL)
            if (!is.null(m)) {
                ci <- confint(m)
                annual <- rbind(annual, data.frame(
                    Year = y, Disorder = psych_labels[col], Group = grp_info$name,
                    Prev = 100 * coef(m)[1],
                    Lower = 100 * ci[1, 1], Upper = 100 * ci[1, 2]
                ))
            }
        }
    }
}

cat(sprintf("Computed %d annual prevalence estimates\n", nrow(annual)))

# Save
write.csv(annual, file.path(outdir, "tables/annual_prevalence.csv"), row.names = FALSE)
cat("Saved tables/annual_prevalence.csv\n")
invisible(NULL)

In [ ]:
# ============================================================
# TREND TESTS: Survey-weighted logistic with YEAR as continuous
# ============================================================
cat("TREND TESTS (survey-weighted logistic, YEAR as continuous predictor)\n\n")
cat(sprintf("%-25s  %8s  %8s  %12s  %10s\n",
            "Disorder", "OR/year", "Lower", "Upper", "p-value"))
cat(paste(rep("-", 70), collapse = ""), "\n")

trend_results <- data.frame()
for (col in names(psych_labels)) {
    fmla <- as.formula(paste0(col, " ~ YEAR"))
    m <- tryCatch(
        svyglm(fmla, design = svy, family = quasibinomial()),
        error = function(e) NULL
    )

    if (!is.null(m)) {
        s <- summary(m)
        coefs <- coef(m)
        ci <- confint(m)
        or_year <- exp(coefs["YEAR"])
        or_lo <- exp(ci["YEAR", 1])
        or_hi <- exp(ci["YEAR", 2])
        p <- s$coefficients["YEAR", "Pr(>|t|)"]
        p_str <- ifelse(p < 0.001, "<0.001", sprintf("%.3f", p))

        cat(sprintf("%-25s  %8.3f  %8.3f  %12.3f  %10s\n",
                    psych_labels[col], or_year, or_lo, or_hi, p_str))

        trend_results <- rbind(trend_results, data.frame(
            Disorder = psych_labels[col],
            OR_per_year = or_year, Lower = or_lo, Upper = or_hi, P = p
        ))
    }
}

write.csv(trend_results, file.path(outdir, "tables/trend_tests.csv"), row.names = FALSE)
cat("\nSaved tables/trend_tests.csv\n")
invisible(NULL)

## Figure 3: Temporal Trends — Key Psychiatric Disorders

In [ ]:
# ============================================================
# FIGURE 3: Trend lines for key disorders (overall)
# ============================================================
key_disorders <- c("Depression", "Anxiety", "Bipolar", "Alcohol",
                   "Drug", "ADHD", "Any psychiatric")
trend_overall <- annual[annual$Group == "Overall" & annual$Disorder %in% key_disorders, ]

p3 <- ggplot(trend_overall, aes(x = Year, y = Prev, color = Disorder)) +
    geom_line(linewidth = 0.8) +
    geom_point(size = 2) +
    geom_ribbon(aes(ymin = Lower, ymax = Upper, fill = Disorder), alpha = 0.1, color = NA) +
    scale_x_continuous(breaks = years) +
    scale_color_brewer(palette = "Set1") +
    scale_fill_brewer(palette = "Set1") +
    labs(x = "Year", y = "Survey-Weighted Prevalence (%)",
         title = "Temporal Trends in Psychiatric Comorbidities Among Epilepsy Hospitalizations",
         subtitle = "NIS 2012-2020 (survey-weighted, 95% CI)",
         color = "Disorder", fill = "Disorder") +
    theme_minimal(base_size = 12) +
    theme(legend.position = "right", panel.grid.minor = element_blank())

print(p3)

ggsave(file.path(outdir, "figures/fig3_trends_overall.pdf"), p3,
       width = 11, height = 7, dpi = 300)
ggsave(file.path(outdir, "figures/fig3_trends_overall.png"), p3,
       width = 11, height = 7, dpi = 300)
cat("Saved figures/fig3_trends_overall.pdf/.png\n")
invisible(NULL)

In [ ]:
# ============================================================
# FIGURE 4: Any psychiatric — Surgery vs Non-Surgical over time
# ============================================================
trend_any <- annual[annual$Disorder == "Any psychiatric" & annual$Group != "Overall", ]

p4 <- ggplot(trend_any, aes(x = Year, y = Prev, color = Group)) +
    geom_line(linewidth = 1) +
    geom_point(size = 2.5) +
    geom_ribbon(aes(ymin = Lower, ymax = Upper, fill = Group), alpha = 0.15, color = NA) +
    scale_x_continuous(breaks = years) +
    scale_color_manual(values = c("Surgery" = "#E63946", "Non-Surgical" = "#457B9D")) +
    scale_fill_manual(values = c("Surgery" = "#E63946", "Non-Surgical" = "#457B9D")) +
    labs(x = "Year", y = "Survey-Weighted Prevalence (%)",
         title = "Any Psychiatric Comorbidity: Surgery vs Non-Surgical Epilepsy",
         subtitle = "NIS 2012-2020 (survey-weighted, 95% CI)",
         color = "Group", fill = "Group") +
    theme_minimal(base_size = 12) +
    theme(legend.position = "bottom", panel.grid.minor = element_blank())

print(p4)

ggsave(file.path(outdir, "figures/fig4_any_psych_trend.pdf"), p4,
       width = 10, height = 6, dpi = 300)
ggsave(file.path(outdir, "figures/fig4_any_psych_trend.png"), p4,
       width = 10, height = 6, dpi = 300)
cat("Saved figures/fig4_any_psych_trend.pdf/.png\n")
invisible(NULL)

## Surgery Utilization Trends

In [ ]:
# ============================================================
# SURGERY UTILIZATION TRENDS
# ============================================================
surg_annual <- data.frame()
for (y in years) {
    svy_y <- subset(svy, YEAR == y)

    for (sg in c("Resective", "VNS", "RNS")) {
        fmla <- as.formula(paste0("~I(surgery_group == '", sg, "')"))
        m <- tryCatch(svymean(fmla, svy_y, na.rm = TRUE), error = function(e) NULL)
        if (!is.null(m)) {
            idx <- 2
            if (length(coef(m)) == 1) idx <- 1
            ci <- confint(m)
            surg_annual <- rbind(surg_annual, data.frame(
                Year = y, Type = sg,
                Prev = 100 * coef(m)[idx],
                Lower = 100 * ci[idx, 1],
                Upper = 100 * ci[idx, 2]
            ))
        }
    }
}

p5 <- ggplot(surg_annual, aes(x = Year, y = Prev, color = Type)) +
    geom_line(linewidth = 0.8) +
    geom_point(size = 2) +
    scale_x_continuous(breaks = years) +
    scale_color_brewer(palette = "Dark2") +
    labs(x = "Year", y = "% of Epilepsy Hospitalizations",
         title = "Epilepsy Surgery Utilization Trends",
         subtitle = "NIS 2012-2020 (survey-weighted)",
         color = "Surgery Type") +
    theme_minimal(base_size = 12) +
    theme(legend.position = "bottom", panel.grid.minor = element_blank())

print(p5)

ggsave(file.path(outdir, "figures/fig5_surgery_trends.pdf"), p5,
       width = 10, height = 6, dpi = 300)
ggsave(file.path(outdir, "figures/fig5_surgery_trends.png"), p5,
       width = 10, height = 6, dpi = 300)
cat("Saved figures/fig5_surgery_trends.pdf/.png\n")
invisible(NULL)